# Construct the GitHub Issues Dataset
### Author: Achintya Kattemalavadi (akattema@berkeley.edu)
This notebook contains the code to collect the GitHub issue and label data used in the multi-label GitHub issues classification project.

### Imports

In [ ]:
import json
import re
from github import Github
from github import Auth
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

### Authenticate and set up GitHub instance

In [ ]:
# using an access token
auth = Auth.Token("GITHUB_KEY")

# Public Web Github
g = Github(auth=auth)

### Get repos

In [ ]:
# Dictionary containing repo objects using the GitHub instance
repos_dict = {
    "flutter": g.get_repo("flutter/flutter"),
    "cdk": g.get_repo("aws/aws-cdk"),
    "camel": g.get_repo("camel-ai/camel"),
    "pub_apis": g.get_repo("public-apis/public-apis"),
    "vscode": g.get_repo("microsoft/vscode"),
    "tensorflow": g.get_repo("tensorflow/tensorflow"),
    "react_native": g.get_repo("facebook/react-native"),
    "pytorch": g.get_repo("pytorch/pytorch"),
    "zephyr": g.get_repo("zephyrproject-rtos/zephyr"),
    "gogs": g.get_repo("gogs/gogs")
}

### Get all possible issue labels for each repo

In [ ]:
labels_dict = {}

# For each repo, get all possible labels
for r_name, r in repos_dict.items():
    labels_dict[r_name] = []
    r_labs = r.get_labels()
    for lab in r_labs:
        labels_dict[r_name].append(lab.name)

# Write to file
with open("data/repo_labels.json", "w") as r_labs_file:
    json.dump(labels_dict, r_labs_file, indent=4)

### Get common labels across repos

We do this through the following method:

1. Use an existing model (`all-MiniLM-L6-v2`) to group lists of semantically similar labels under representitive labels
2. Filter out any labels that do not belong to at least 3 repos in the dataset
3. Manually check the remaining label groups and filter out any that will not work for the task

In [ ]:
def preprocess(word):
    """Normalize a word by:
    - Converting to lowercase
    - Removing punctuation and symbols
    - Replacing spaces and special characters with underscores
    """
    word = word.lower()  # Convert to lowercase
    word = re.sub(r"[^\w\s]", "", word)  # Remove punctuation and symbols
    word = re.sub(r"\s+", "_", word)  # Replace spaces with underscores
    return word

In [ ]:
# Load the sentence transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

def find_common_terms_from_json(labels_dict):
    """Read JSON file, extract lists, and group semantically similar words."""

    # Reverse map words to original keys (handling duplicate words under multiple keys)
    word_to_keys = defaultdict(set)
    for key, words in labels_dict.items():
        for word in words:
            normalized_word = preprocess(word)
            word_to_keys[normalized_word].add(key)

    # Extract all unique words
    all_words = list(word_to_keys.keys())

    if not all_words:  # Handle empty input
        return {}

    # Get embeddings
    embeddings = model.encode(all_words)

    # Compute similarity matrix
    sim_matrix = cosine_similarity(embeddings)

    grouped_words = defaultdict(set)
    for i in range(len(all_words)):
        for j in range(i + 1, len(all_words)):
            if sim_matrix[i, j] > 0.75:  # Adjust threshold for better grouping
                grouped_words[all_words[i]].add(all_words[j])
                grouped_words[all_words[j]].add(all_words[i])

    unique_groups = []
    seen = set()
    for word, synonyms in grouped_words.items():
        if word not in seen:
            group = {word} | synonyms
            unique_groups.append(group)
            seen.update(group)

    # Create dictionary with representative word as key, and a sub-dictionary as value
    result_dict = {}
    for group in unique_groups:
        rep_word = min(group, key=len)  # Shortest word as representative
        result_dict[rep_word] = {
            word: list(word_to_keys[word]) for word in group  # Convert sets to lists
        }

    return result_dict

In [ ]:
common_words_dict = find_common_terms_from_json(labels_dict)

with open("data/common_labels.json", "w") as f:
    json.dump(common_words_dict, f, indent=4)

In [ ]:
def filter_multi_source_words(nested_dict, num_unique_src=3):
    """
    Filters the nested dictionary to keep only entries where the subdictionary 
    contains 3 or more unique sources.

    :param nested_dict: The output dictionary from `find_common_terms_from_json`
    :return: A filtered dictionary containing only terms with at least 3 unique sources.
    """
    filtered_dict = {}

    for rep_word, sub_dict in nested_dict.items():
        # Extract unique source keys from the subdictionary
        unique_sources = set(source for sources in sub_dict.values() for source in sources)

        # Keep only if there are 3 or more unique sources
        if len(unique_sources) >= num_unique_src:
            filtered_dict[rep_word] = sub_dict

    return filtered_dict

In [ ]:
filtered_common_words = filter_multi_source_words(common_words_dict)

with open("data/common_labels_diff_only.json", "w") as f:
    json.dump(filtered_common_words, f, indent=4)

### Create a reverse index for labels to common names

We will need to load data from the manually-edited common labels JSON. We will use this reverse index to replace the labels with the common term when processing the GitHub issues.

In [ ]:
common_labels = None
common_labels_file = "data/common_labels_final.json" 
with open(common_labels_file, "r", encoding="utf-8") as f:
    common_labels = json.load(f)

# Extract just the dictionary containing common label names as keys and lists of similar labels as values
cmn_labs_clean = {
    lab: list(similar_labs.keys()) for lab, similar_labs in common_labels.items()
}

common_labels_reverse = {}

# Create the reverse index
for lab, sim_labs in cmn_labs_clean.items():
    for sim_lab in sim_labs:
        common_labels_reverse[sim_lab] = lab

with open("data/common_labels_final_reverse.json", "w") as f:
    json.dump(common_labels_reverse, f, indent=4)

### Actually construct the dataset

We will need to read closed issues from each repository and replace label names with the common names we created before.

First, we create a multi-layer dictionary with the following format:
```
{
    "repo_1": {
        "161430": {
            "title": "\ModalBottomSheet's DragHandle...",
            "body": "When using a screeneader...",
            "comments": [
                "Thanks for the report...",
                "This thread has been..."
            ],
            "labels": [
                "good_first_issue",
                "accessibility"
            ]
        }
    }
}
```

In [ ]:
raw_data_filename = "data/pre_processed_issues_all.json"
all_issues = {}
count_until_chkpt = 50

# For each repo
for r_name, r in repos_dict.items():

    print(f"Repo: {r_name}")

    if not r_name in all_issues:
        all_issues[r_name] = {}

    # Pull all closed issues
    r_iss_l = r.get_issues(state="closed")
    for iss in r_iss_l:
        try:
            if str(iss.number) in all_issues[r_name]:
                print(f"Issue {iss.number} already recorded, skipping.")
                continue

            i_labs = set(
                common_labels_reverse[preprocessed_name]
                for l in iss.labels
                if (preprocessed_name := preprocess(l.name)) in common_labels_reverse
            )

            if len(i_labs) == 0:
                print(f"Issue {iss.number} has no labels, skipping.")
                continue

            print(f"Adding issue {iss.number}...")

            iss_dict = {
                "title": iss.title,
                "body": iss.body,
                "comments": [c.body for c in iss.get_comments()],
                "labels": list(i_labs)
            }
            all_issues[r_name][iss.number] = iss_dict
            count_until_chkpt -= 1

        except Exception as e:
            print(f"Skipping issue {iss.number} due to problem: {e}")
            continue
        
        if not count_until_chkpt:
            print(f"Reached checkpoint, writing to {raw_data_filename}")
            with open(raw_data_filename, "w") as f:
                json.dump(all_issues, f, indent=4)
            count_until_chkpt = 50

with open(raw_data_filename, "w") as f:
    json.dump(all_issues, f, indent=4)

Finally, we need to flatten out the dataset so it's a list of issue dictionaries.

In [ ]:
issues_list = []

for repo, issues in all_issues.items():

    for iss_id, iss_dat in issues.items():

        iss_restruct = {
            "repo_name": repo,
            "issue_id": iss_id
        }

        for k, v in iss_dat.items():
            if k == "labels": continue
            elif k == "comments": iss_restruct[k] = str(v)
            else: iss_restruct[k] = v
        
        for lab in common_labels:
            iss_restruct[lab] = True if lab in iss_dat.get("labels", []) else False
        
        issues_list.append(iss_restruct)

dataset_restruct_file = "data/dataset_reformatted_final.json" 
with open(dataset_restruct_file, "w") as f:
    json.dump(issues_list, f, indent=4)